In [1]:
%pip install transformers sentencepiece langdetect

  Using cached langdetect-1.0.9-py3-none-any.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 3.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langdetect]

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from langdetect import detect

In [3]:
text = "What Data Science courses are available?"

language = detect(text)

print("Detected Language:", language)

Detected Language: fr


In [4]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

In [5]:
text = "I want to learn Data Science and Machine Learning courses at GUVI."

language = detect(text)

print("Detected Language:", language)

Detected Language: en


In [6]:
# Test a non-English sentence

text = "எனக்கு டேட்டா சயின்ஸ் படிக்க வேண்டும்"

language = detect(text)

print("Detected Language:", language)

Detected Language: ta


In [7]:
def detect_language(text):
    try:
        return detect(text)
    except:
        return "en"

In [8]:
print(detect_language("I want to learn Python"))
print(detect_language("எனக்கு பைதான் கற்க வேண்டும்"))

en
ta


### Load the multilingual translation model

Tamil → English
English → Tamil
and other supported languages

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [11]:
translation_model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(
    translation_model_name
)

translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    translation_model_name
)

print("Translation model loaded successfully!")

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 69309.44it/s]


Translation model loaded successfully!


### Create the language-code mapping

In [12]:
LANGUAGE_MAP = {
    "en": "eng_Latn",   # English
    "ta": "tam_Taml",   # Tamil
    "hi": "hin_Deva",   # Hindi
    "te": "tel_Telu",   # Telugu
    "kn": "kan_Knda",   # Kannada
    "ml": "mal_Mlym"    # Malayalam
}

print(LANGUAGE_MAP)

{'en': 'eng_Latn', 'ta': 'tam_Taml', 'hi': 'hin_Deva', 'te': 'tel_Telu', 'kn': 'kan_Knda', 'ml': 'mal_Mlym'}


This will eventually allow our pipeline to do:

Tamil question
     ↓

langdetect → ta
     ↓

LANGUAGE_MAP → tam_Taml
     ↓

NLLB
     ↓

English question

### Create the translation function

In [13]:
def translate_text(text, source_lang, target_lang):
    # Convert langdetect codes to NLLB language codes
    source_code = LANGUAGE_MAP[source_lang]
    target_code = LANGUAGE_MAP[target_lang]

    # Set source language
    tokenizer.src_lang = source_code

    # Convert text into tokens
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    )

    # Generate translated tokens
    translated_tokens = translation_model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_code),
        max_length=512
    )

    # Convert tokens back into readable text
    translated_text = tokenizer.batch_decode(
        translated_tokens,
        skip_special_tokens=True
    )[0]

    return translated_text

In [14]:
# Test Tamil → English translation

tamil_text = "எனக்கு டேட்டா சயின்ஸ் படிக்க வேண்டும்"

english_translation = translate_text(
    tamil_text,
    source_lang="ta",
    target_lang="en"
)

print("Tamil:", tamil_text)
print("English:", english_translation)

Tamil: எனக்கு டேட்டா சயின்ஸ் படிக்க வேண்டும்
English: I want to study data science.


In [15]:
# Test English → Tamil translation

english_text = "GUVI offers courses in Data Science and Machine Learning."

tamil_translation = translate_text(
    english_text,
    source_lang="en",
    target_lang="ta"
)

print("English:", english_text)
print("Tamil:", tamil_translation)

English: GUVI offers courses in Data Science and Machine Learning.
Tamil: GUVI தரவு அறிவியல் மற்றும் இயந்திர கற்றல் ஆகியவற்றில் படிப்புகளை வழங்குகிறது.


### Combine language detection + translation

In [16]:
def translate_to_english(text):
    detected_lang = detect_language(text)

    print("Detected Language:", detected_lang)

    # Already English — no translation required
    if detected_lang == "en":
        return text

    # Check whether the language is supported
    if detected_lang not in LANGUAGE_MAP:
        return "Unsupported language"

    # Translate detected language → English
    translated_text = translate_text(
        text,
        source_lang=detected_lang,
        target_lang="en"
    )

    return translated_text

This gives us the first half of our multilingual pipeline

User Question
     ↓

Detect Language
     ↓

English? ── Yes → Keep original
     ↓ No

Translate to English
     ↓

English Query

In [17]:
# Test automatic Tamil → English flow

user_text = "எனக்கு டேட்டா சயின்ஸ் பாடநெறி பற்றி தகவல் வேண்டும்"

english_text = translate_to_english(user_text)

print("Original:", user_text)
print("English:", english_text)

Detected Language: ta
Original: எனக்கு டேட்டா சயின்ஸ் பாடநெறி பற்றி தகவல் வேண்டும்
English: I want information about the data science course


In [18]:
# Test automatic English handling

user_text = "Tell me about the Data Science course at GUVI"

english_text = translate_to_english(user_text)

print("Original:", user_text)
print("English:", english_text)

Detected Language: en
Original: Tell me about the Data Science course at GUVI
English: Tell me about the Data Science course at GUVI


In [19]:
# Create the reverse translation function

def translate_from_english(text, target_lang):
    # English user → no translation needed
    if target_lang == "en":
        return text

    # Check whether the target language is supported
    if target_lang not in LANGUAGE_MAP:
        return text

    # Translate English → user's language
    translated_text = translate_text(
        text,
        source_lang="en",
        target_lang=target_lang
    )

    return translated_text

In [20]:
# Test English → original-language response

english_answer = "GUVI offers a Data Science course with practical learning and career support."

final_answer = translate_from_english(
    english_answer,
    target_lang="ta"
)

print("English Answer:", english_answer)
print("Tamil Answer:", final_answer)

English Answer: GUVI offers a Data Science course with practical learning and career support.
Tamil Answer: GUVI தரவு அறிவியல் பாடத்திட்டத்தை நடைமுறை கற்றல் மற்றும் தொழில் ஆதரவுடன் வழங்குகிறது.
